In [ ]:
import time

import pandas as pd

from src.data.load_raw import load_raw_data
from src.data.preprocess import (
    extract_xy,
    merge_features_and_labels,
    standardize_splits,
)
from src.data.split import split_labeled_transactions
from src.evaluation.metrics import calculate_binary_metrics
from src.evaluation.threshold import find_best_f1_threshold
from src.models.mlp import (
    build_mlp,
)

: 

In [2]:
features_df, classes_df, edges_df = load_raw_data()

transactions_df = merge_features_and_labels(
    features_df,
    classes_df,
)

splits = split_labeled_transactions(transactions_df)

print("Features:", features_df.shape)
print("Classes:", classes_df.shape)
print("Edges:", edges_df.shape)

print("\nTrain:", len(splits.train))
print("Validation:", len(splits.validation))
print("Test:", len(splits.test))

print("\nTrain labels:")
print(splits.train["target"].value_counts())

print("\nValidation labels:")
print(splits.validation["target"].value_counts())

print("\nTest labels:")
print(splits.test["target"].value_counts())

Features: (203769, 167)
Classes: (203769, 2)
Edges: (234355, 2)

Train: 29894
Validation: 7829
Test: 8841

Train labels:
target
0    26432
1     3462
Name: count, dtype: Int64

Validation labels:
target
0    7154
1     675
Name: count, dtype: Int64

Test labels:
target
0    8433
1     408
Name: count, dtype: Int64


In [3]:
x_train, y_train = extract_xy(
    splits.train,
    feature_set="local",
)

x_validation, y_validation = extract_xy(
    splits.validation,
    feature_set="local",
)

x_test, y_test = extract_xy(
    splits.test,
    feature_set="local",
)

(
    scaler,
    x_train_scaled,
    x_validation_scaled,
    x_test_scaled,
) = standardize_splits(
    x_train,
    x_validation,
    x_test,
)

In [4]:
results = []


def run_experiment(
    model_name,
    model,
    x_train,
    y_train,
    x_validation,
    y_validation,
    x_test,
    y_test,
):
    start_time = time.perf_counter()

    model.fit(
        x_train,
        y_train,
    )

    training_time = time.perf_counter() - start_time

    validation_scores = model.predict_proba(
        x_validation,
    )[:, 1]

    threshold = find_best_f1_threshold(
        y_validation,
        validation_scores,
    )

    test_scores = model.predict_proba(
        x_test,
    )[:, 1]

    metrics = calculate_binary_metrics(
        y_test,
        test_scores,
        threshold=threshold,
    )

    metrics.update(
        {
            "model": model_name,
            "feature_set": "local",
            "training_time_seconds": training_time,
        }
    )

    results.append(metrics)

    return model, metrics

In [5]:
mlp_model, mlp_metrics = run_experiment(
    model_name="MLP",
    model=build_mlp(4),
    x_train=x_train,
    y_train=y_train,
    x_validation=x_validation,
    y_validation=y_validation,
    x_test=x_test,
    y_test=y_test
)

In [6]:
def run_feature_set_experiment(
    feature_set,
    model_name,
    model,
):
    x_train, y_train = extract_xy(
        splits.train,
        feature_set=feature_set,
    )

    x_validation, y_validation = extract_xy(
        splits.validation,
        feature_set=feature_set,
    )

    x_test, y_test = extract_xy(
        splits.test,
        feature_set=feature_set,
    )

    start_time = time.perf_counter()

    model.fit(x_train, y_train)

    training_time = time.perf_counter() - start_time

    validation_scores = model.predict_proba(
        x_validation,
    )[:, 1]

    threshold = find_best_f1_threshold(
        y_validation,
        validation_scores,
    )

    test_scores = model.predict_proba(
        x_test,
    )[:, 1]

    metrics = calculate_binary_metrics(
        y_test,
        test_scores,
        threshold=threshold,
    )

    metrics.update(
        {
            "model": model_name,
            "feature_set": feature_set,
            "training_time_seconds": training_time,
        }
    )

    results.append(metrics)

    return model, metrics

In [13]:
mlp_all_model, mlp_all_metrics = run_feature_set_experiment(
    feature_set="all",
    model_name="MLP3",
    model=build_mlp(3)
)

In [14]:
results_df = pd.DataFrame(results)

columns = [
    "model",
    "feature_set",
    "pr_auc",
    "roc_auc",
    "precision_illicit",
    "recall_illicit",
    "f1_illicit",
    "threshold",
    "training_time_seconds",
]

results_df[columns].sort_values(
    by="pr_auc",
    ascending=False,
).style.format(
    {
        "pr_auc": "{:.4f}",
        "roc_auc": "{:.4f}",
        "precision_illicit": "{:.4f}",
        "recall_illicit": "{:.4f}",
        "f1_illicit": "{:.4f}",
        "threshold": "{:.2f}",
        "training_time_seconds": "{:.2f}",
    }
)

display(results_df)

,pr_auc,roc_auc,precision_illicit,recall_illicit,f1_illicit,confusion_matrix,threshold,model,feature_set,training_time_seconds
0,0.374209,0.806564,0.654545,0.441176,0.527086,"[[8338, 95], [228, 180]]",0.48,MLP,local,33.371880
1,0.361891,0.844454,0.522472,0.455882,0.486911,"[[8263, 170], [222, 186]]",0.20,MLP,all,22.908171
2,0.492493,0.842219,0.699248,0.455882,0.551929,"[[8353, 80], [222, 186]]",0.49,MLP2,all,20.251226
3,0.388122,0.842259,0.497382,0.465686,0.481013,"[[8241, 192], [218, 190]]",0.21,MLP3,all,89.678142
4,0.456893,0.855591,0.700375,0.458333,0.554074,"[[8353, 80], [221, 187]]",0.15,MLP3,all,14.455741
